"""
ResNeSt101 - İki Aşamalı Transfer Learning Eğitim Scripti
=========================================================
Aşama 1: Backbone Freeze  → sadece classifier katmanı eğitilir (5 epoch)
Aşama 2: Fine-Tuning      → tüm ağ düşük LR ile eğitilir (5 epoch)
 
Augmentation sırası (Ablation Study):
  - MixUp + CutMix (karma)  → varsayılan
  - Sonra sadece MixUp veya sadece CutMix için AUGMENTATION_MODE değiştir
 
Kullanım:
  pip install torch torchvision timm scikit-learn matplotlib seaborn tqdm
  python train_resnest101.py
"""

In [1]:
import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

c:\Users\ardac\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR       = r"C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split"
OUTPUT_DIR     = r"C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101_cutmix"
IMG_SIZE       = 224
BATCH_SIZE     = 16        # RTX 3050 4GB için güvenli değer
NUM_WORKERS    = 0         # Windows CMD için 0 olmalı         # CPU thread sayısı
PIN_MEMORY     = True
 
# Epoch sayıları (düşük tutuldu - hızlı test için)
FREEZE_EPOCHS  = 5         # Aşama 1: Backbone dondurulmuş
FINETUNE_EPOCHS = 5        # Aşama 2: Fine-tune
 
# Learning rate
FREEZE_LR      = 1e-3      # Aşama 1 LR
FINETUNE_LR    = 1e-5      # Aşama 2 LR (çok düşük!)
 
# Augmentation modu: "both" | "mixup" | "cutmix" | "none"
AUGMENTATION_MODE = "cutmix"
 
# MixUp / CutMix parametreleri
MIXUP_ALPHA    = 0.4
CUTMIX_ALPHA   = 1.0
MIXUP_PROB     = 0.5       # "both" modunda MixUp seçilme olasılığı
 
SEED           = 42
SUBSET_RATIO   = 0.4       # Her epoch train setinin %40'ını kullan (hız için)

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
 
set_seed(SEED)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Kullanılan cihaz: {device}")
if device.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
 
os.makedirs(OUTPUT_DIR, exist_ok=True)

✅ Kullanılan cihaz: cuda
   GPU: NVIDIA GeForce RTX 3050 Laptop GPU
   VRAM: 4.3 GB


In [5]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
 
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [6]:
# ─────────────────────────────────────────────
# SAFE DATASET - Hatalı dosyaları atlar
# ─────────────────────────────────────────────
from torchvision.datasets import ImageFolder
from torchvision.datasets.folder import default_loader
 
class SafeImageFolder(ImageFolder):
    """Açılamayan dosyaları sessizce atlayan ImageFolder"""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Başlangıçta tüm dosyaları tara, açılamayanları çıkar
        print(f"   🔍 Dosyalar doğrulanıyor: {self.root}")
        valid_samples = []
        skipped = 0
        for path, label in tqdm(self.samples, desc="   Taranıyor", leave=False):
            try:
                with open(path, 'rb') as f:
                    f.read(10)  # Sadece açılabilir mi kontrol et
                valid_samples.append((path, label))
            except (FileNotFoundError, OSError):
                skipped += 1
        if skipped > 0:
            print(f"   ⚠️  {skipped} dosya atlandı (bulunamadı/açılamadı)")
        self.samples = valid_samples
        self.targets = [s[1] for s in valid_samples]
        print(f"   ✅ Geçerli dosya sayısı: {len(self.samples)}")

In [7]:
# ─────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────
print("\n📂 Veri seti yükleniyor...")
train_dataset = SafeImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
val_dataset   = SafeImageFolder(os.path.join(DATA_DIR, "val"),   transform=val_test_transform)
test_dataset  = SafeImageFolder(os.path.join(DATA_DIR, "test"),  transform=val_test_transform)
 
NUM_CLASSES = len(train_dataset.classes)
print(f"   Sınıf sayısı : {NUM_CLASSES}")
print(f"   Train        : {len(train_dataset)} görsel")
print(f"   Validation   : {len(val_dataset)} görsel")
print(f"   Test         : {len(test_dataset)} görsel")
 


📂 Veri seti yükleniyor...
   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\train


   ⚠️  1 dosya atlandı (bulunamadı/açılamadı)
   ✅ Geçerli dosya sayısı: 34019
   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\val


   ✅ Geçerli dosya sayısı: 2043
   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\test


   ✅ Geçerli dosya sayısı: 1979
   Sınıf sayısı : 47
   Train        : 34019 görsel
   Validation   : 2043 görsel
   Test         : 1979 görsel


In [8]:
# ─────────────────────────────────────────────
# WEIGHTED RANDOM SAMPLER
# ─────────────────────────────────────────────
def get_weighted_sampler(dataset):
    targets = [s[1] for s in dataset.samples]
    class_counts = np.bincount(targets)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[t] for t in targets]
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )
    return sampler
 
sampler = get_weighted_sampler(train_dataset)
 
# Subset: her epoch sadece SUBSET_RATIO kadar veri kullan
subset_size = int(len(train_dataset) * SUBSET_RATIO)
subset_sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor([sampler.weights[i] for i in range(len(train_dataset))]),
    num_samples=subset_size,
    replacement=True
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=subset_sampler,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print(f"   Her epoch: {subset_size} görsel / {subset_size//BATCH_SIZE} batch (toplam {len(train_dataset)} görselin %{int(SUBSET_RATIO*100)}'i)")
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

   Her epoch: 13607 görsel / 850 batch (toplam 34019 görselin %40'i)


In [9]:
# ─────────────────────────────────────────────
# MixUp / CutMix Fonksiyonları
# ─────────────────────────────────────────────
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam
 
def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    _, _, H, W = x.size()
    cut_rat = np.sqrt(1 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam
 
def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [10]:
# ─────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────
print("\n🔧 ResNeSt101 modeli yükleniyor (timm)...")
model = timm.create_model('resnest101e', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)
print(f"   Model yüklendi. Parametre sayısı: {sum(p.numel() for p in model.parameters()):,}")


🔧 ResNeSt101 modeli yükleniyor (timm)...
   Model yüklendi. Parametre sayısı: 46,322,319


In [11]:
# ─────────────────────────────────────────────
# FREEZE / UNFREEZE Yardımcıları
# ─────────────────────────────────────────────
def freeze_backbone(model):
    """Sadece sınıflandırıcı katmanı eğitilebilir bırak"""
    for name, param in model.named_parameters():
        if 'fc' in name or 'head' in name or 'classifier' in name:
            param.requires_grad = True
        else:
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   ❄️  Backbone donduruldu. Eğitilebilir parametre: {trainable:,}")
 
def unfreeze_all(model):
    """Tüm parametreleri eğitilebilir yap"""
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   🔥 Tüm model açıldı. Eğitilebilir parametre: {trainable:,}")

In [12]:
# ─────────────────────────────────────────────
# METRİK HESAPLAMA
# ─────────────────────────────────────────────
def evaluate(model, loader, criterion, phase="val"):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
 
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
 
    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
 
    return avg_loss, acc, prec, rec, f1, all_preds, all_labels

In [13]:
# ─────────────────────────────────────────────
# CONFUSION MATRIX & RAPOR KAYDET
# ─────────────────────────────────────────────
def save_confusion_matrix(labels, preds, class_names, save_path, title="Confusion Matrix"):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(28, 24))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, annot_kws={"size": 6})
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Tahmin", fontsize=10)
    ax.set_ylabel("Gerçek", fontsize=10)
    plt.xticks(rotation=90, fontsize=6)
    plt.yticks(rotation=0, fontsize=6)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   💾 Confusion matrix kaydedildi: {save_path}")
 
def save_metrics_plot(history, save_path, title="Eğitim Grafiği"):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(title, fontsize=14)
 
    metrics = [
        ('train_loss', 'val_loss', 'Loss'),
        ('train_acc',  'val_acc',  'Accuracy'),
        ('train_f1',   'val_f1',   'F1 Score'),
        ('train_prec', 'val_prec', 'Precision'),
        ('train_rec',  'val_rec',  'Recall'),
    ]
    for i, (tr_key, val_key, label) in enumerate(metrics):
        ax = axes[i // 3][i % 3]
        ax.plot(epochs, history[tr_key], 'b-o', label='Train')
        ax.plot(epochs, history[val_key], 'r-o', label='Val')
        ax.set_title(label)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True)
    axes[1][2].axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"   💾 Grafik kaydedildi: {save_path}")

In [14]:
# ─────────────────────────────────────────────
# EĞİTİM FONKSİYONU
# ─────────────────────────────────────────────
def train_phase(model, train_loader, val_loader, optimizer, scheduler,
                criterion, num_epochs, phase_name, aug_mode):
 
    history = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc',
                                 'train_f1','val_f1','train_prec','val_prec',
                                 'train_rec','val_rec']}
    best_val_f1 = 0.0
    best_model_path = os.path.join(OUTPUT_DIR, f"best_{phase_name}.pth")
 
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        all_preds, all_labels = [], []
        start = time.time()
 
        skipped_batches = 0
        pbar = tqdm(train_loader, desc=f"[{phase_name}] Epoch {epoch}/{num_epochs}")
        for batch_data in pbar:
            try:
                images, labels = batch_data
                images, labels = images.to(device), labels.to(device)
 
                # Augmentation uygula
                use_aug = aug_mode != "none"
                if use_aug:
                    if aug_mode == "both":
                        use_mixup = random.random() < MIXUP_PROB
                    elif aug_mode == "mixup":
                        use_mixup = True
                    else:
                        use_mixup = False  # cutmix
 
                    if use_mixup:
                        images, y_a, y_b, lam = mixup_data(images, labels, MIXUP_ALPHA)
                    else:
                        images, y_a, y_b, lam = cutmix_data(images, labels, CUTMIX_ALPHA)
 
                    optimizer.zero_grad()
                    outputs = model(images)
                    loss = mixed_criterion(criterion, outputs, y_a, y_b, lam)
                else:
                    optimizer.zero_grad()
                    outputs = model(images)
                    loss = criterion(outputs, labels)
 
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
 
                total_loss += loss.item()
                preds = outputs.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            except Exception:
                skipped_batches += 1
                continue
 
        if skipped_batches > 0:
            print(f"{skipped_batches} batch atlandı (dosya erişim hatası)")
 
        if scheduler:
            scheduler.step()
 
        # Train metrikleri
        tr_loss = total_loss / len(train_loader)
        tr_acc  = accuracy_score(all_labels, all_preds)
        tr_f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        tr_prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
        tr_rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
 
        # Val metrikleri
        val_loss, val_acc, val_prec, val_rec, val_f1, _, _ = evaluate(model, val_loader, criterion)
 
        elapsed = time.time() - start
        print(f"\n📊 [{phase_name}] Epoch {epoch}/{num_epochs} ({elapsed:.0f}s)")
        print(f"   Train → Loss:{tr_loss:.4f}  Acc:{tr_acc:.4f}  F1:{tr_f1:.4f}  Prec:{tr_prec:.4f}  Rec:{tr_rec:.4f}")
        print(f"   Val   → Loss:{val_loss:.4f}  Acc:{val_acc:.4f}  F1:{val_f1:.4f}  Prec:{val_prec:.4f}  Rec:{val_rec:.4f}")
 
        # History kaydet
        history['train_loss'].append(tr_loss);  history['val_loss'].append(val_loss)
        history['train_acc'].append(tr_acc);    history['val_acc'].append(val_acc)
        history['train_f1'].append(tr_f1);      history['val_f1'].append(val_f1)
        history['train_prec'].append(tr_prec);  history['val_prec'].append(val_prec)
        history['train_rec'].append(tr_rec);    history['val_rec'].append(val_rec)
 
        # En iyi modeli kaydet
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"   ✅ En iyi model kaydedildi (Val F1: {best_val_f1:.4f})")
 
    return history, best_model_path

In [15]:
# ─────────────────────────────────────────────
# TEST DEĞERLENDİRME
# ─────────────────────────────────────────────
def test_evaluation(model, test_loader, criterion, class_names, phase_label, aug_label):
    print(f"\n{'='*60}")
    print(f"🧪 TEST DEĞERLENDİRMESİ - {phase_label} | Aug: {aug_label}")
    print(f"{'='*60}")
    _, acc, prec, rec, f1, preds, labels = evaluate(model, test_loader, criterion, "test")
    print(f"   Accuracy  : {acc:.4f}")
    print(f"   Precision : {prec:.4f}")
    print(f"   Recall    : {rec:.4f}")
    print(f"   F1 Score  : {f1:.4f}")
 
    # Confusion matrix
    cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{phase_label}_{aug_label}.png")
    save_confusion_matrix(labels, preds, class_names, cm_path,
                          title=f"Confusion Matrix - {phase_label} | {aug_label}")
 
    # Classification report
    report = classification_report(labels, preds, target_names=class_names, zero_division=0)
    report_path = os.path.join(OUTPUT_DIR, f"classification_report_{phase_label}_{aug_label}.txt")
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(f"Phase: {phase_label} | Augmentation: {aug_label}\n\n")
        f.write(report)
    print(f"   💾 Classification report: {report_path}")
 
    return acc, prec, rec, f1

In [19]:
import sys
!{sys.executable} -m pip install grad-cam


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
# ─────────────────────────────────────────────
# GRAD-CAM DEĞERLENDİRMESİ
# ─────────────────────────────────────────────
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import cv2
 
GRADCAM_DIR = os.path.join(OUTPUT_DIR, "gradcam")
os.makedirs(GRADCAM_DIR, exist_ok=True)
 
SAMPLES_PER_CLASS = 2   # Her sınıftan kaç görsel
 
# Grad-CAM için normalize edilmemiş transform (görsel gösterimi için)
vis_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])
 
norm_transform = transforms.Normalize(
    [0.485, 0.456, 0.406],
    [0.229, 0.224, 0.225]
)
 
def get_target_layer(model):
    """ResNeSt101 için son conv katmanını döndür"""
    return [model.layer4[-1]]
 
def run_gradcam(model, model_name, dataset, class_names):
    """Verilen model için her sınıftan SAMPLES_PER_CLASS görsel üzerinde Grad-CAM uygula"""
    model.eval()
    target_layers = get_target_layer(model)
    cam = GradCAM(model=model, target_layers=target_layers)
 
    save_dir = os.path.join(GRADCAM_DIR, model_name)
    os.makedirs(save_dir, exist_ok=True)
 
    # Her sınıftan örnek topla
    class_samples = {i: [] for i in range(len(class_names))}
    for path, label in dataset.samples:
        if len(class_samples[label]) < SAMPLES_PER_CLASS:
            class_samples[label].append(path)
        if all(len(v) >= SAMPLES_PER_CLASS for v in class_samples.values()):
            break
 
    print(f"\n🎨 [{model_name}] Grad-CAM oluşturuluyor...")
    for class_idx, paths in tqdm(class_samples.items(), desc=model_name):
        class_name = class_names[class_idx]
        fig, axes = plt.subplots(len(paths), 3, figsize=(12, 4 * len(paths)))
        if len(paths) == 1:
            axes = [axes]
        fig.suptitle(f"{model_name} | Sınıf: {class_name}", fontsize=11)
 
        for row, path in enumerate(paths):
            try:
                # Orijinal görseli yükle
                from PIL import Image as PILImage
                pil_img = PILImage.open(path).convert("RGB")
                pil_img = pil_img.resize((IMG_SIZE, IMG_SIZE))
                rgb_img = np.array(pil_img).astype(np.float32) / 255.0
 
                # Model için tensor
                input_tensor = norm_transform(vis_transform(pil_img)).unsqueeze(0).to(device)
 
                # Tahmin
                with torch.no_grad():
                    output = model(input_tensor)
                pred_idx = output.argmax(dim=1).item()
                pred_name = class_names[pred_idx]
                confidence = torch.softmax(output, dim=1)[0][pred_idx].item()
 
                # Grad-CAM
                targets = [ClassifierOutputTarget(class_idx)]
                grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
                grayscale_cam = grayscale_cam[0]
                cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
 
                # Doğru mu yanlış mı
                is_correct = (pred_idx == class_idx)
                border_color = "green" if is_correct else "red"
                status = "✅ Doğru" if is_correct else f"❌ Yanlış → {pred_name}"
 
                # Görselleri çiz
                axes[row][0].imshow(pil_img)
                axes[row][0].set_title("Orijinal", fontsize=9)
                axes[row][0].axis("off")
 
                axes[row][1].imshow(grayscale_cam, cmap="jet")
                axes[row][1].set_title("Grad-CAM Isı Haritası", fontsize=9)
                axes[row][1].axis("off")
 
                axes[row][2].imshow(cam_image)
                axes[row][2].set_title(
                    f"{status}\nGüven: {confidence:.2%}", fontsize=9,
                    color=border_color
                )
                axes[row][2].axis("off")
                for spine in axes[row][2].spines.values():
                    spine.set_edgecolor(border_color)
                    spine.set_linewidth(3)
 
            except Exception as e:
                print(f"   ⚠️  {path} atlandı: {e}")
                continue
 
        plt.tight_layout()
        safe_name = class_name.replace("/", "_").replace("\\", "_")
        save_path = os.path.join(save_dir, f"{safe_name}.png")
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
        plt.close()
 
    print(f"   💾 Kaydedildi: {save_dir}")

In [22]:
# ─────────────────────────────────────────────
# MODELLERİ YÜKLE VE GRAD-CAM ÇALIŞTIR
# ─────────────────────────────────────────────
FREEZE_PATH   = os.path.join(OUTPUT_DIR, "best_freeze.pth")
FINETUNE_PATH = os.path.join(OUTPUT_DIR, "best_finetune.pth")
 
# Test dataseti kullan (augmentation yok, temiz görseller)
gradcam_dataset = SafeImageFolder(
    os.path.join(DATA_DIR, "test"),
    transform=val_test_transform
)
 
print("\n" + "="*60)
print("🔍 GRAD-CAM BAŞLIYOR")
print("="*60)
print(f"   Her sınıftan {SAMPLES_PER_CLASS} görsel x {len(train_dataset.classes)} sınıf")
print(f"   Toplam: ~{SAMPLES_PER_CLASS * len(train_dataset.classes)} görsel x 2 model")
 
# Backbone Freeze modeli
print("\n⚡ Backbone Freeze modeli yükleniyor...")
model.load_state_dict(torch.load(FREEZE_PATH, map_location=device))
run_gradcam(model, "Phase1_BackboneFreeze", gradcam_dataset, train_dataset.classes)
 
# Fine-Tune modeli
print("\n🔥 Fine-Tune modeli yükleniyor...")
model.load_state_dict(torch.load(FINETUNE_PATH, map_location=device))
run_gradcam(model, "Phase2_FineTune", gradcam_dataset, train_dataset.classes)
 
print("\n" + "="*60)
print(f"✅ GRAD-CAM TAMAMLANDI!")
print(f"   Sonuçlar: {GRADCAM_DIR}")
print(f"   Phase1_BackboneFreeze/ → Freeze modeli görselleri")
print(f"   Phase2_FineTune/       → Fine-tune modeli görselleri")
print("="*60)

   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\test


   ✅ Geçerli dosya sayısı: 1979

🔍 GRAD-CAM BAŞLIYOR
   Her sınıftan 2 görsel x 47 sınıf
   Toplam: ~94 görsel x 2 model

⚡ Backbone Freeze modeli yükleniyor...

🎨 [Phase1_BackboneFreeze] Grad-CAM oluşturuluyor...


Phase1_BackboneFreeze: 100%|██████████| 47/47 [00:35<00:00,  1.31it/s]


   💾 Kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101\gradcam\Phase1_BackboneFreeze

🔥 Fine-Tune modeli yükleniyor...

🎨 [Phase2_FineTune] Grad-CAM oluşturuluyor...


Phase2_FineTune: 100%|██████████| 47/47 [01:00<00:00,  1.28s/it]

   💾 Kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101\gradcam\Phase2_FineTune

✅ GRAD-CAM TAMAMLANDI!
   Sonuçlar: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101\gradcam
   Phase1_BackboneFreeze/ → Freeze modeli görselleri
   Phase2_FineTune/       → Fine-tune modeli görselleri


In [16]:
# ─────────────────────────────────────────────
# ANA AKIŞ
# ─────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
class_names = train_dataset.classes
 
print(f"\n{'='*60}")
print(f"🚀 EĞİTİM BAŞLIYOR")
print(f"   Model          : ResNeSt101e")
print(f"   Augmentation   : {AUGMENTATION_MODE}")
print(f"   Freeze Epochs  : {FREEZE_EPOCHS}")
print(f"   Finetune Epochs: {FINETUNE_EPOCHS}")
print(f"   Batch Size     : {BATCH_SIZE}")
print(f"{'='*60}\n")


🚀 EĞİTİM BAŞLIYOR
   Model          : ResNeSt101e
   Augmentation   : cutmix
   Freeze Epochs  : 5
   Finetune Epochs: 5
   Batch Size     : 16



In [17]:
# ──────────────────────────────
# AŞAMA 1: BACKBONE FREEZE
# ──────────────────────────────
print("\n" + "="*60)
print("⚡ AŞAMA 1: BACKBONE FREEZE")
print("="*60)
freeze_backbone(model)
 
optimizer_freeze = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FREEZE_LR, weight_decay=1e-4
)
scheduler_freeze = optim.lr_scheduler.CosineAnnealingLR(optimizer_freeze, T_max=FREEZE_EPOCHS)
 
history_freeze, best_freeze_path = train_phase(
    model, train_loader, val_loader,
    optimizer_freeze, scheduler_freeze,
    criterion, FREEZE_EPOCHS,
    phase_name="freeze",
    aug_mode=AUGMENTATION_MODE
)
 
# Grafik kaydet
save_metrics_plot(history_freeze,
                  os.path.join(OUTPUT_DIR, "grafik_freeze.png"),
                  title=f"Aşama 1: Backbone Freeze | Aug: {AUGMENTATION_MODE}")
 
# Aşama 1 test değerlendirmesi
print("\n📌 Aşama 1 tamamlandı → Test seti üzerinde değerlendiriliyor...")
model.load_state_dict(torch.load(best_freeze_path))
acc1, prec1, rec1, f1_1 = test_evaluation(
    model, test_loader, criterion, class_names,
    phase_label="Phase1_Freeze", aug_label=AUGMENTATION_MODE
)


⚡ AŞAMA 1: BACKBONE FREEZE
   ❄️  Backbone donduruldu. Eğitilebilir parametre: 3,673,999


[freeze] Epoch 1/5:   0%|          | 0/851 [00:00<?, ?it/s]

[freeze] Epoch 1/5: 100%|██████████| 851/851 [46:22<00:00,  3.27s/it, loss=2.4249]  



📊 [freeze] Epoch 1/5 (2846s)
   Train → Loss:2.5501  Acc:0.3639  F1:0.3581  Prec:0.3589  Rec:0.3626
   Val   → Loss:1.0316  Acc:0.6960  F1:0.6458  Prec:0.6717  Rec:0.6632
   ✅ En iyi model kaydedildi (Val F1: 0.6458)


[freeze] Epoch 2/5: 100%|██████████| 851/851 [06:08<00:00,  2.31it/s, loss=2.0532]


...


📊 [freeze] Epoch 3/5 (390s)
   Train → Loss:1.9487  Acc:0.5280  F1:0.5220  Prec:0.5216  Rec:0.5245
   Val   → Loss:0.8242  Acc:0.7567  F1:0.7172  Prec:0.7342  Rec:0.7263
   ✅ En iyi model kaydedildi (Val F1: 0.7172)


[freeze] Epoch 4/5: 100%|██████████| 851/851 [05:57<00:00,  2.38it/s, loss=1.5727]



📊 [freeze] Epoch 4/5 (383s)
   Train → Loss:1.8558  Acc:0.5587  F1:0.5552  Prec:0.5549  Rec:0.5578
   Val   → Loss:0.7600  Acc:0.7739  F1:0.7396  Prec:0.7466  Rec:0.7443
   ✅ En iyi model kaydedildi (Val F1: 0.7396)


[freeze] Epoch 5/5: 100%|██████████| 851/851 [05:56<00:00,  2.39it/s, loss=1.0752]



📊 [freeze] Epoch 5/5 (381s)
   Train → Loss:1.8285  Acc:0.5588  F1:0.5558  Prec:0.5554  Rec:0.5578
   Val   → Loss:0.7280  Acc:0.7900  F1:0.7567  Prec:0.7643  Rec:0.7607
   ✅ En iyi model kaydedildi (Val F1: 0.7567)
   💾 Grafik kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101_cutmix\grafik_freeze.png

📌 Aşama 1 tamamlandı → Test seti üzerinde değerlendiriliyor...

🧪 TEST DEĞERLENDİRMESİ - Phase1_Freeze | Aug: cutmix
   Accuracy  : 0.7721
   Precision : 0.7389
   Recall    : 0.7388
   F1 Score  : 0.7318
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101_cutmix\confusion_matrix_Phase1_Freeze_cutmix.png
   💾 Classification report: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101_cutmix\classification_report_Phase1_Freeze_cutmix.txt


In [18]:
# ──────────────────────────────
# AŞAMA 1 SONUÇLARINI YUKLE (önceden kaydedildi)
# ──────────────────────────────
best_freeze_path = os.path.join(OUTPUT_DIR, "best_freeze.pth")
print("Aşama 1 modeli yükleniyor:", best_freeze_path)
model.load_state_dict(torch.load(best_freeze_path, map_location=device))
print("✅ Model yüklendi!")
 
# Aşama 1 test değerlendirmesi (isteğe bağlı - atlamak için yoruma alın)
print("Aşama 1 test değerlendirmesi yapılıyor...")
acc1, prec1, rec1, f1_1 = test_evaluation(
    model, test_loader, criterion, class_names,
    phase_label="Phase1_Freeze", aug_label=AUGMENTATION_MODE
)

Aşama 1 modeli yükleniyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101\best_freeze.pth
✅ Model yüklendi!
Aşama 1 test değerlendirmesi yapılıyor...

🧪 TEST DEĞERLENDİRMESİ - Phase1_Freeze | Aug: both
   Accuracy  : 0.7928
   Precision : 0.7621
   Recall    : 0.7591
   F1 Score  : 0.7545
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101\confusion_matrix_Phase1_Freeze_both.png
   💾 Classification report: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101\classification_report_Phase1_Freeze_both.txt


In [18]:
# ──────────────────────────────
# AŞAMA 2: FINE-TUNING
# ──────────────────────────────
print("\n" + "="*60)
print("🔥 AŞAMA 2: FINE-TUNING (Backbone açılıyor)")
print("="*60)
unfreeze_all(model)
 
optimizer_finetune = optim.AdamW(
    model.parameters(),
    lr=FINETUNE_LR, weight_decay=1e-4
)
scheduler_finetune = optim.lr_scheduler.CosineAnnealingLR(optimizer_finetune, T_max=FINETUNE_EPOCHS)
 
history_finetune, best_finetune_path = train_phase(
    model, train_loader, val_loader,
    optimizer_finetune, scheduler_finetune,
    criterion, FINETUNE_EPOCHS,
    phase_name="finetune",
    aug_mode=AUGMENTATION_MODE
)
 
# Grafik kaydet
save_metrics_plot(history_finetune,
                  os.path.join(OUTPUT_DIR, "grafik_finetune.png"),
                  title=f"Aşama 2: Fine-Tuning | Aug: {AUGMENTATION_MODE}")
 
# Aşama 2 test değerlendirmesi
print("\n📌 Aşama 2 tamamlandı → Test seti üzerinde değerlendiriliyor...")
model.load_state_dict(torch.load(best_finetune_path))
acc2, prec2, rec2, f1_2 = test_evaluation(
    model, test_loader, criterion, class_names,
    phase_label="Phase2_FineTune", aug_label=AUGMENTATION_MODE
)


🔥 AŞAMA 2: FINE-TUNING (Backbone açılıyor)
   🔥 Tüm model açıldı. Eğitilebilir parametre: 46,322,319


[finetune] Epoch 1/5: 100%|██████████| 851/851 [18:51<00:00,  1.33s/it, loss=1.6780]



📊 [finetune] Epoch 1/5 (1162s)
   Train → Loss:1.7083  Acc:0.5907  F1:0.5880  Prec:0.5876  Rec:0.5906
   Val   → Loss:0.7134  Acc:0.7900  F1:0.7551  Prec:0.7573  Rec:0.7633
   ✅ En iyi model kaydedildi (Val F1: 0.7551)


[finetune] Epoch 2/5: 100%|██████████| 851/851 [18:08<00:00,  1.28s/it, loss=2.5409]



📊 [finetune] Epoch 2/5 (1119s)
   Train → Loss:1.6292  Acc:0.6044  F1:0.6021  Prec:0.6022  Rec:0.6034
   Val   → Loss:0.6863  Acc:0.8013  F1:0.7688  Prec:0.7708  Rec:0.7763
   ✅ En iyi model kaydedildi (Val F1: 0.7688)


...


📊 [finetune] Epoch 3/5 (982s)
   Train → Loss:1.5596  Acc:0.6312  F1:0.6290  Prec:0.6291  Rec:0.6308
   Val   → Loss:0.6765  Acc:0.8032  F1:0.7712  Prec:0.7747  Rec:0.7740
   ✅ En iyi model kaydedildi (Val F1: 0.7712)


[finetune] Epoch 4/5: 100%|██████████| 851/851 [15:20<00:00,  1.08s/it, loss=2.4116]



📊 [finetune] Epoch 4/5 (945s)
   Train → Loss:1.5695  Acc:0.6267  F1:0.6248  Prec:0.6249  Rec:0.6273
   Val   → Loss:0.6509  Acc:0.8101  F1:0.7780  Prec:0.7806  Rec:0.7819
   ✅ En iyi model kaydedildi (Val F1: 0.7780)


[finetune] Epoch 5/5: 100%|██████████| 851/851 [15:14<00:00,  1.07s/it, loss=1.9261]



📊 [finetune] Epoch 5/5 (939s)
   Train → Loss:1.5375  Acc:0.6481  F1:0.6458  Prec:0.6459  Rec:0.6475
   Val   → Loss:0.6392  Acc:0.8116  F1:0.7775  Prec:0.7792  Rec:0.7810
   💾 Grafik kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101_cutmix\grafik_finetune.png

📌 Aşama 2 tamamlandı → Test seti üzerinde değerlendiriliyor...

🧪 TEST DEĞERLENDİRMESİ - Phase2_FineTune | Aug: cutmix
   Accuracy  : 0.8110
   Precision : 0.7775
   Recall    : 0.7768
   F1 Score  : 0.7720
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101_cutmix\confusion_matrix_Phase2_FineTune_cutmix.png
   💾 Classification report: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101_cutmix\classification_report_Phase2_FineTune_cutmix.txt


In [19]:
# ──────────────────────────────
# KARŞILAŞTIRMA TABLOSU
# ──────────────────────────────
print("\n" + "="*60)
print("📈 SONUÇ KARŞILAŞTIRMASI")
print("="*60)
print(f"{'Aşama':<30} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
print("-"*60)
print(f"{'Phase 1 (Freeze)':<30} {acc1:>8.4f} {prec1:>8.4f} {rec1:>8.4f} {f1_1:>8.4f}")
print(f"{'Phase 2 (Fine-Tune)':<30} {acc2:>8.4f} {prec2:>8.4f} {rec2:>8.4f} {f1_2:>8.4f}")
print("="*60)
 
# Özet dosyası kaydet
summary_path = os.path.join(OUTPUT_DIR, f"summary_{AUGMENTATION_MODE}.txt")
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(f"ResNeSt101 Eğitim Özeti\n")
    f.write(f"Augmentation: {AUGMENTATION_MODE}\n")
    f.write(f"{'='*60}\n")
    f.write(f"{'Aşama':<30} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}\n")
    f.write(f"{'-'*60}\n")
    f.write(f"{'Phase 1 (Freeze)':<30} {acc1:>8.4f} {prec1:>8.4f} {rec1:>8.4f} {f1_1:>8.4f}\n")
    f.write(f"{'Phase 2 (Fine-Tune)':<30} {acc2:>8.4f} {prec2:>8.4f} {rec2:>8.4f} {f1_2:>8.4f}\n")
 
print(f"\n✅ Tüm sonuçlar kaydedildi: {OUTPUT_DIR}")
print(f"   Özet dosyası: {summary_path}")
print("\n🎉 EĞİTİM TAMAMLANDI!")


📈 SONUÇ KARŞILAŞTIRMASI
Aşama                               Acc     Prec      Rec       F1
------------------------------------------------------------
Phase 1 (Freeze)                 0.7721   0.7389   0.7388   0.7318
Phase 2 (Fine-Tune)              0.8110   0.7775   0.7768   0.7720

✅ Tüm sonuçlar kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101_cutmix
   Özet dosyası: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnest101_cutmix\summary_cutmix.txt

🎉 EĞİTİM TAMAMLANDI!
